# Statistics in Python

Walkthrough of the examples from [Scipy Lecture Notes — Statistics in Python](https://scipy-lectures.org/packages/statistics/index.html) (Gaël Varoquaux).

**Requirements:** numpy, scipy, matplotlib, pandas, statsmodels, seaborn

Each code cell follows an inline example from the tutorial. Comments explain what the cell is doing and why.


## 3.1.1 Data representation and interaction

### 3.1.1.1 Data as a table

Statistical analysis often starts from a 2D table: rows = observations, columns = attributes/features.

We use `examples/brain_size.csv` (Willerman et al. 1991): brain size (MRI), body measures, and IQ scores.


### 3.1.1.2 The pandas data-frame

#### Creating dataframes: reading data files or converting arrays

**Reading from a CSV file.** The file mixes numeric and categorical columns. Missing values are marked with `"."`.


In [1]:
# Import pandas — the Python equivalent of a spreadsheet / R data.frame
import pandas

# Read the brain-size study CSV.
# sep=';'  → fields are semicolon-separated (European-style CSV)
# na_values="." → treat "." as missing (NaN), so Weight for subject 2 is usable in stats
data = pandas.read_csv('examples/brain_size.csv', sep=';', na_values=".")
data


,Unnamed: 0,Gender,FSIQ,VIQ,PIQ,Weight,Height,MRI_Count
0,1,Female,133,132,124,118.0,64.5,816932
1,2,Male,140,150,124,NaN,72.5,1001121
2,3,Male,139,123,150,143.0,73.3,1038437
3,4,Male,133,129,128,172.0,68.8,965353
4,5,Female,137,132,134,147.0,65.0,951545
5,6,Female,99,90,110,146.0,69.0,928799
6,7,Female,138,136,131,138.0,64.5,991305
7,8,Female,92,90,98,175.0,66.0,854258
8,9,Male,89,93,84,134.0,66.3,904858
9,10,Male,133,114,147,172.0,68.8,955466


**Creating from arrays.** A DataFrame can also be built from a dictionary of 1D arrays/lists.


In [2]:
# numpy provides array creation and math helpers
import numpy as np

# Evenly spaced points from -6 to 6 (20 values)
t = np.linspace(-6, 6, 20)
# Element-wise sine and cosine of those points
sin_t = np.sin(t)
cos_t = np.cos(t)


In [3]:
# Bundle the three arrays into a named-column DataFrame (like a small table)
pandas.DataFrame({'t': t, 'sin': sin_t, 'cos': cos_t})


,t,sin,cos
0,-6.000000,0.279415,0.960170
1,-5.368421,0.792419,0.609977
2,-4.736842,0.999701,0.024451
3,-4.105263,0.821291,-0.570509
4,-3.473684,0.326021,-0.945363
5,-2.842105,-0.295030,-0.955488
6,-2.210526,-0.802257,-0.596979
7,-1.578947,-0.999967,-0.008151
8,-0.947368,-0.811882,0.583822
9,-0.315789,-0.310567,0.950551


#### Manipulating data

`data` behaves like R's dataframe: named columns, mixed types, flexible selection.


In [4]:
# Shape: (n_rows, n_columns) — should be 40 subjects × 8 columns
data.shape    # 40 rows and 8 columns

# Column names (includes an unnamed index column from the CSV)
data.columns  # It has columns

# Select one column by name (returns a Series)
print(data['Gender'])  # Columns can be addressed by name

# Boolean filter: keep Female rows, then take VIQ, then mean
# Simpler selector
data[data['Gender'] == 'Female']['VIQ'].mean()


0     Female
1       Male
2       Male
3       Male
4     Female
5     Female
6     Female
7     Female
8       Male
9       Male
10    Female
11      Male
12      Male
13    Female
14    Female
15    Female
16    Female
17      Male
18    Female
19      Male
20      Male
21      Male
22    Female
23      Male
24    Female
25      Male
26    Female
27      Male
28    Female
29    Female
30    Female
31      Male
32      Male
33      Male
34    Female
35    Female
36      Male
37    Female
38      Male
39      Male
Name: Gender, dtype: object


109.45

**groupby:** split a dataframe on values of a categorical variable.


In [5]:
# Split the frame into Male / Female groups
groupby_gender = data.groupby('Gender')

# For each gender, print the mean Verbal IQ (VIQ)
for gender, value in groupby_gender['VIQ']:
    print((gender, value.mean()))


('Female', 109.45)
('Male', 115.25)


In [6]:
# Apply mean() to every numeric column within each Gender group
# Useful summary of how males and females differ on average
groupby_gender.mean()


,Unnamed: 0,FSIQ,VIQ,PIQ,Weight,Height,MRI_Count
Gender,,,,,,,
Female,19.65,111.9,109.45,110.45,137.200000,65.765000,862654.6
Male,21.35,115.0,115.25,111.60,166.444444,71.431579,954855.4


#### Plotting data

Pandas plotting helpers (matplotlib under the hood) make quick exploratory plots.

> **Note:** The tutorial used `pandas.tools.plotting`, which was removed. Use `pandas.plotting` instead.


In [7]:
# Modern import path (pandas.tools.plotting is deprecated/removed)
from pandas import plotting

# Scatter-matrix: pairwise relationships among body size and MRI volume
plotting.scatter_matrix(data[['Weight', 'Height', 'MRI_Count']])


array([[<Axes: xlabel='Weight', ylabel='Weight'>,
        <Axes: xlabel='Height', ylabel='Weight'>,
        <Axes: xlabel='MRI_Count', ylabel='Weight'>],
       [<Axes: xlabel='Weight', ylabel='Height'>,
        <Axes: xlabel='Height', ylabel='Height'>,
        <Axes: xlabel='MRI_Count', ylabel='Height'>],
       [<Axes: xlabel='Weight', ylabel='MRI_Count'>,
        <Axes: xlabel='Height', ylabel='MRI_Count'>,
        <Axes: xlabel='MRI_Count', ylabel='MRI_Count'>]], dtype=object)

In [8]:
# Scatter-matrix among the three IQ measures (should be strongly related)
plotting.scatter_matrix(data[['PIQ', 'VIQ', 'FSIQ']])


array([[<Axes: xlabel='PIQ', ylabel='PIQ'>,
        <Axes: xlabel='VIQ', ylabel='PIQ'>,
        <Axes: xlabel='FSIQ', ylabel='PIQ'>],
       [<Axes: xlabel='PIQ', ylabel='VIQ'>,
        <Axes: xlabel='VIQ', ylabel='VIQ'>,
        <Axes: xlabel='FSIQ', ylabel='VIQ'>],
       [<Axes: xlabel='PIQ', ylabel='FSIQ'>,
        <Axes: xlabel='VIQ', ylabel='FSIQ'>,
        <Axes: xlabel='FSIQ', ylabel='FSIQ'>]], dtype=object)

## 3.1.2 Hypothesis testing: comparing two groups

For simple tests we use `scipy.stats`.


In [9]:
# scipy.stats provides classical hypothesis tests (t-tests, Wilcoxon, etc.)
from scipy import stats


### 3.1.2.1 Student's t-test: the simplest statistical test

#### 1-sample t-test: testing the value of a population mean

`ttest_1samp` asks whether the population mean of a sample is equal to a given value (here, 0).


In [10]:
# H0: mean VIQ == 0. With IQ scores ~100, we expect a very small p-value.
# Returns (t-statistic, p-value)
stats.ttest_1samp(data['VIQ'], 0)


Ttest_1sampResult(statistic=30.088099970849328, pvalue=1.3289196468728067e-28)

#### 2-sample t-test: testing for difference across populations

Male and female mean VIQ looked different. Is that difference significant?


In [11]:
# Extract VIQ for each gender as separate Series
female_viq = data[data['Gender'] == 'Female']['VIQ']
male_viq = data[data['Gender'] == 'Male']['VIQ']

# Independent (unpaired) two-sample t-test: H0 = equal means
stats.ttest_ind(female_viq, male_viq)


Ttest_indResult(statistic=-0.7726161723275011, pvalue=0.44452876778583217)

### 3.1.2.2 Paired tests: repeated measurements on the same individuals

FSIQ, VIQ, and PIQ are three IQ measures on the *same* people. An unpaired 2-sample test ignores that pairing.


In [12]:
# Naive unpaired test comparing FSIQ vs PIQ as if independent samples
# This confounds subject-to-subject variability
stats.ttest_ind(data['FSIQ'], data['PIQ'])


Ttest_indResult(statistic=0.465637596380964, pvalue=0.6427725009414841)

In [13]:
# Paired (repeated-measures) t-test: accounts for within-subject pairing
stats.ttest_rel(data['FSIQ'], data['PIQ'])


Ttest_relResult(statistic=1.7842019405859857, pvalue=0.08217263818364236)

In [14]:
# Equivalent formulation: 1-sample t-test on the paired differences vs 0
stats.ttest_1samp(data['FSIQ'] - data['PIQ'], 0)


Ttest_1sampResult(statistic=1.7842019405859857, pvalue=0.08217263818364236)

T-tests assume Gaussian errors. The Wilcoxon signed-rank test relaxes that assumption.


In [15]:
# Non-parametric paired alternative to the paired t-test
stats.wilcoxon(data['FSIQ'], data['PIQ'])


WilcoxonResult(statistic=274.5, pvalue=0.10659492713506856)

## 3.1.3 Linear models, multiple factors, and analysis of variance

### 3.1.3.1 "formulas" to specify statistical models in Python

#### A simple linear regression

Model: \( y = \text{intercept} + \text{coef}\, x + e \). Fit with OLS (ordinary least squares) via statsmodels, then test that the slope is nonzero.

First, simulate data from a known linear model.


In [16]:
import numpy as np

# Predictor values evenly spaced on [-5, 5]
x = np.linspace(-5, 5, 20)

# Reproducible random noise
np.random.seed(1)

# True model: y = -5 + 3*x + Gaussian noise (sd=4)
# normal distributed noise
y = -5 + 3 * x + 4 * np.random.normal(size=x.shape)

# Create a data frame containing all the relevant variables
data = pandas.DataFrame({'x': x, 'y': y})


In [17]:
# Formula API: "y ~ x" means regress y on x (with intercept by default)
from statsmodels.formula.api import ols

# Fit ordinary least squares and store the results object
model = ols("y ~ x", data).fit()


In [18]:
# Full regression report: coefficients, SEs, t-tests, R², diagnostics
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.804
Model:                            OLS   Adj. R-squared:                  0.794
Method:                 Least Squares   F-statistic:                     74.03
Date:                Mon, 27 Jul 2026   Prob (F-statistic):           8.56e-08
Time:                        16:34:41   Log-Likelihood:                -57.988
No. Observations:                  20   AIC:                             120.0
Df Residuals:                      18   BIC:                             122.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -5.5335      1.036     -5.342      0.0

#### Categorical variables: comparing groups or multiple categories

Reload the brain-size data and compare male vs female VIQ with a linear model (equivalent to a 2-sample t-test).


In [19]:
# Reload brain_size — previous cells overwrote `data` with the simulated regression frame
data = pandas.read_csv('examples/brain_size.csv', sep=';', na_values=".")


In [20]:
# VIQ ~ Gender: Gender is treated as categorical; +1 forces an intercept
# Intercept = mean VIQ for the reference level (Female); Gender[T.Male] = male−female difference
model = ols("VIQ ~ Gender + 1", data).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    VIQ   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                    0.5969
Date:                Mon, 27 Jul 2026   Prob (F-statistic):              0.445
Time:                        16:34:41   Log-Likelihood:                -182.42
No. Observations:                  40   AIC:                             368.8
Df Residuals:                      38   BIC:                             372.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        109.4500      5.308     20.

In [21]:
# Force categorical coding explicitly with C(...)
# Useful when a column is stored as integers but should be treated as categories
model = ols('VIQ ~ C(Gender)', data).fit()


**Link to t-tests between different FSIQ and PIQ.** Build a long-form table so IQ *type* is a categorical factor.


In [22]:
# Stack FSIQ and PIQ into long format: one row per (person, IQ type)
data_fisq = pandas.DataFrame({'iq': data['FSIQ'], 'type': 'fsiq'})
data_piq = pandas.DataFrame({'iq': data['PIQ'], 'type': 'piq'})
data_long = pandas.concat((data_fisq, data_piq))
print(data_long)

# OLS on long data: type[T.piq] coefficient mirrors the unpaired FSIQ vs PIQ contrast
model = ols("iq ~ type", data_long).fit()
print(model.summary())


     iq  type
0   133  fsiq
1   140  fsiq
2   139  fsiq
3   133  fsiq
4   137  fsiq
..  ...   ...
35  128   piq
36  124   piq
37   94   piq
38   74   piq
39   89   piq

[80 rows x 2 columns]
                            OLS Regression Results                            
Dep. Variable:                     iq   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                    0.2168
Date:                Mon, 27 Jul 2026   Prob (F-statistic):              0.643
Time:                        16:34:41   Log-Likelihood:                -364.35
No. Observations:                  80   AIC:                             732.7
Df Residuals:                      78   BIC:                             737.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  c

In [23]:
# Same contrast via an independent t-test — t and p should match the OLS type effect
stats.ttest_ind(data['FSIQ'], data['PIQ'])


Ttest_indResult(statistic=0.465637596380964, pvalue=0.6427725009414841)

### 3.1.3.2 Multiple Regression: including multiple factors

Iris example: sepal/petal sizes are related, but is there also a species effect?


In [24]:
# Load Fisher's iris data (sepal/petal lengths and species name)
data = pandas.read_csv('examples/iris.csv')

# Predict sepal_width from species (categorical) AND petal_length (continuous)
model = ols('sepal_width ~ name + petal_length', data).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            sepal_width   R-squared:                       0.478
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     44.63
Date:                Mon, 27 Jul 2026   Prob (F-statistic):           1.58e-20
Time:                        16:34:41   Log-Likelihood:                -38.185
No. Observations:                 150   AIC:                             84.37
Df Residuals:                     146   BIC:                             96.41
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              2.9813      0

### 3.1.3.3 Post-hoc hypothesis testing: analysis of variance (ANOVA)

Test whether versicolor and virginica differ in the model above via a contrast / F-test:
`name[T.versicolor] - name[T.virginica]`.


In [25]:
# Contrast vector matches parameter order in the model:
# [Intercept, name[T.versicolor], name[T.virginica], petal_length]
# [0, 1, -1, 0] tests versicolor coeff − virginica coeff == 0
print(model.f_test([0, 1, -1, 0]))


<F test: F=array([[3.24533535]]), p=0.07369058781700577, df_denom=146, df_num=1>


## 3.1.4 More visualization: seaborn for statistical exploration

Load the CPS wage dataset (Berndt 1991 / CMU CPS_85_Wages), then explore with seaborn.


In [26]:
# --- Load wages data (as in the tutorial's full figure examples) ---
import os
import urllib.request

wages_path = 'examples/wages.txt'
if not os.path.exists(wages_path):
    urllib.request.urlretrieve(
        'https://lib.stat.cmu.edu/datasets/CPS_85_Wages',
        wages_path,
    )

# Column names from the CPS_85_Wages documentation
names = [
    'EDUCATION: Number of years of education',
    'SOUTH: 1=Person lives in South, 0=Person lives elsewhere',
    'SEX: 1=Female, 0=Male',
    'EXPERIENCE: Number of years of work experience',
    'UNION: 1=Union member, 0=Not union member',
    'WAGE: Wage (dollars per hour)',
    'AGE: years',
    'RACE: 1=Other, 2=Hispanic, 3=White',
    'OCCUPATION: 1=Management, 2=Sales, 3=Clerical, 4=Service, 5=Professional, 6=Other',
    'SECTOR: 0=Other, 1=Manufacturing, 2=Construction',
    'MARR: 0=Unmarried,  1=Married',
]
short_names = [n.split(':')[0] for n in names]

# skiprows/skipfooter skip the CMU documentation header/footer around the data table
data = pandas.read_csv(
    wages_path,
    skiprows=27,
    skipfooter=6,
    sep=None,
    header=None,
    engine='python',  # required for skipfooter
)
data.columns = short_names

# Log10 wages: wages often scale multiplicatively
data['WAGE'] = np.log10(data['WAGE'])

print(data)


     EDUCATION  SOUTH  SEX  EXPERIENCE  UNION      WAGE  AGE  RACE  \
0            8      0    1          21      0  0.707570   35     2   
1            9      0    1          42      0  0.694605   57     3   
2           12      0    0           1      0  0.824126   19     3   
3           12      0    0           4      0  0.602060   22     3   
4           12      0    0          17      0  0.875061   35     3   
..         ...    ...  ...         ...    ...       ...  ...   ...   
529         18      0    0           5      0  1.055378   29     3   
530         12      0    1          33      0  0.785330   51     1   
531         17      0    1          25      1  1.366423   48     1   
532         12      1    0          13      1  1.298416   31     3   
533         16      0    0          33      0  1.186956   55     3   

     OCCUPATION  SECTOR  MARR  
0             6       1     1  
1             6       1     1  
2             6       1     0  
3             6       0     0  

### 3.1.4.1 Pairplot: scatter matrices


In [27]:
# seaborn: statistical visualization layered on matplotlib + pandas
import seaborn

# Pairwise scatterplots with univariate regression fits on the diagonal/off-diagonal
seaborn.pairplot(data, vars=['WAGE', 'AGE', 'EDUCATION'], kind='reg')


In [28]:
# Same pairplot, colored by SEX (1=Female, 0=Male) to reveal group structure
seaborn.pairplot(data, vars=['WAGE', 'AGE', 'EDUCATION'], kind='reg', hue='SEX')


Importing seaborn changes matplotlib defaults. Reset with `plt.rcdefaults()` if needed.


In [29]:
# Restore matplotlib's default rcParams after seaborn's style changes
from matplotlib import pyplot as plt
plt.rcdefaults()


### 3.1.4.2 lmplot: plotting a univariate regression


In [30]:
# Scatter + linear regression of (log10) wage vs years of education
seaborn.lmplot(y='WAGE', x='EDUCATION', data=data)


## 3.1.5 Testing for interactions

Do wages increase more with education for males than females? Model an interaction term.


In [31]:
# Prepare a slim frame with string gender labels for formula-friendly categoricals
# (tutorial uses lowercase column names education / gender / wage)
import statsmodels.formula.api as sm

wage_df = pandas.DataFrame({
    'education': data['EDUCATION'],
    # SEX in the raw file: 1=Female, 0=Male → map to strings
    'gender': np.choose(data['SEX'].astype(int), ['male', 'female']),
    'wage': data['WAGE'],  # already log10-transformed above
})

# Interaction model: education * gender expands to main effects + interaction
# education:gender[T.male] tests whether the education slope differs by gender
result = sm.ols(
    formula='wage ~ education + gender + education * gender',
    data=wage_df,
).fit()
print(result.summary())


                            OLS Regression Results                            
Dep. Variable:                   wage   R-squared:                       0.198
Model:                            OLS   Adj. R-squared:                  0.194
Method:                 Least Squares   F-statistic:                     43.72
Date:                Mon, 27 Jul 2026   Prob (F-statistic):           2.94e-25
Time:                        16:34:47   Log-Likelihood:                 88.503
No. Observations:                 534   AIC:                            -169.0
Df Residuals:                     530   BIC:                            -151.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## Take-home messages

- Hypothesis testing and p-values quantify the significance of an effect / difference.
- Formulas (with categorical variables) express rich links in your data.
- Visualizing data and fitting simple models builds intuition.
- Conditioning (adding factors that explain variation) changes interpretation.
